# Music Annotation Project
## Notebook 75A: Collar-Exposed and Hard-Negative Stratified Reanalysis

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook is a **read-only post-hoc reanalysis** of one completed Notebook 75 run. It does **not** regenerate C0/C1/C2 pages, alter any arm, change any threshold, or modify `v8_8_operator.py`.

### Purpose

Notebook 75 pooled two different page roles:

1. **COLLAR_EXPOSED** pages, where C1/C2 actually restored pixels and therefore contribute evidence about Ring 2 / Ring 1 preservation.
2. **ZERO_AUTHORITY / HARD-NEGATIVE CONTROL** pages, where no collar was exposed and C1/C2 correctly had nothing to restore.

Notebook 75A separates these roles, recalculates page-clustered uncertainty only on the collar-exposed population, and reports the zero-intervention controls separately.

### Role rule

The notebook first looks for an **explicit hard-negative / sparse / control flag** in the available development/roster manifests. If found, that explicit role is authoritative.

If no explicit role field is available, the notebook does **not** silently call every zero-exposure page a hard negative. It reports those pages as `ZERO_EXPOSURE_UNCLASSIFIED`, while also checking whether their C1/C2 outputs are exactly zero.

### Historical integrity

The completed Notebook 75 run remains unchanged. Notebook 75A writes a new reanalysis directory with its own tables, charts, manifest and ZIP review package.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy matplotlib seaborn

## 1. Configuration and source-run resolution

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, zipfile, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB75_ROOT = ROOT / 'notebook75_v88_collar_preservation_counterfactual'
OUTPUT_ROOT = ROOT / 'notebook75a_stratified_reanalysis'
SOURCE_NB75_RUN_OVERRIDE = None

SEED = 42
BOOTSTRAP_RESAMPLES = 10000
PAGE201_POLICY = 'REPORT_SEPARATELY'   # REPORT_SEPARATELY | EXCLUDE_INVALID | KEEP
KNOWN_INVALID_PAGES = {'Page201'}      # applied only when PAGE201_POLICY == EXCLUDE_INVALID

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook75a_stratified_reanalysis'
OUT = OUTPUT_ROOT / RUN
TABLES = OUT / 'tables'
VISUALS = OUT / 'visuals'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
for p in [OUT, TABLES, VISUALS, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)

def require(cond, msg):
    if not bool(cond):
        raise RuntimeError(msg)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

def atomic_json(path, value):
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def latest_complete(root):
    hits=[]
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in [
                'tables/results_by_page.csv',
                'tables/pooled_arm_summary.csv',
                'tables/incremental_ring1_over_ring2.csv',
                'prediction_lock/prediction_lock.json',
                'notebook75_verdict.json',
            ]):
                hits.append(p)
    return max(hits, key=lambda p:p.stat().st_mtime) if hits else None

SOURCE = Path(SOURCE_NB75_RUN_OVERRIDE) if SOURCE_NB75_RUN_OVERRIDE else latest_complete(NB75_ROOT)
require(SOURCE is not None, 'No complete Notebook 75 run found.')
print('Source Notebook 75:', SOURCE)
print('Notebook 75A output:', OUT)

## 2. Load Notebook 75 evidence and verify it has not changed

In [ ]:
required = {
    'results_by_page': SOURCE/'tables'/'results_by_page.csv',
    'pooled_arm_summary': SOURCE/'tables'/'pooled_arm_summary.csv',
    'incremental_ring1': SOURCE/'tables'/'incremental_ring1_over_ring2.csv',
    'results_by_source_group': SOURCE/'tables'/'results_by_source_group.csv',
    'prediction_lock': SOURCE/'prediction_lock'/'prediction_lock.json',
    'verdict': SOURCE/'notebook75_verdict.json',
}
for name,path in required.items():
    require(path.exists(), f'Missing Notebook 75 artefact: {name}: {path}')

page_df = pd.read_csv(required['results_by_page'])
pooled75 = pd.read_csv(required['pooled_arm_summary'])
incremental75 = pd.read_csv(required['incremental_ring1'])
source_group75 = pd.read_csv(required['results_by_source_group'])
lock75 = json.loads(required['prediction_lock'].read_text())
verdict75 = json.loads(required['verdict'].read_text())

required_cols = {
    'page_key','page','population','source_group','arm','restored_pixels',
    'true_print_recovered','annotation_retained','truth_uncertain_retained',
    'utility','catastrophic_components'
}
missing = required_cols - set(page_df.columns)
require(not missing, f'results_by_page.csv missing fields: {sorted(missing)}')

# Paired population is the only truth-scored population for this reanalysis.
paired = page_df[page_df.population.astype(str).eq('PAIRED_DEVELOPMENT')].copy()
require(len(paired)>0, 'Notebook 75 contains no PAIRED_DEVELOPMENT rows.')
require(set(paired.arm.unique()) == {'C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2'},
        f'Unexpected paired arms: {sorted(paired.arm.unique())}')
require(not paired.duplicated(['page_key','arm']).any(), 'Duplicate page_key/arm rows found.')

source_integrity = {
    'source_run': str(SOURCE),
    'results_by_page_sha256': sha256_file(required['results_by_page']),
    'pooled_arm_summary_sha256': sha256_file(required['pooled_arm_summary']),
    'incremental_ring1_sha256': sha256_file(required['incremental_ring1']),
    'prediction_lock_sha256': sha256_file(required['prediction_lock']),
    'notebook75_verdict_sha256': sha256_file(required['verdict']),
    'source_prediction_lock_id': lock75.get('lock_id'),
}
atomic_json(AUDIT/'source_notebook75_identity.json', source_integrity)
print('Paired Notebook 75 rows:', len(paired), 'pages:', paired.page_key.nunique())

## 3. Resolve explicit page roles where available

The project record requires hard-negative and sparse-print material to remain role-separated. This cell therefore searches likely manifest snapshots for explicit role fields before using exposure as a descriptive fallback.

In [ ]:
MANIFEST_CANDIDATES = [
    SOURCE/'inputs'/'development_score_manifest.csv',
    ROOT/'development_score_manifest.csv',
]
# The Notebook 74 source run is often recorded in Notebook 75's lock/verdict lineage.
source_nb74 = None
for obj in [lock75, verdict75]:
    text = json.dumps(obj, default=str)
    # no path parsing assumption here; direct manifest candidates below are safer

role_manifest = None
role_manifest_path = None
for p in MANIFEST_CANDIDATES:
    if p.exists():
        q = pd.read_csv(p, dtype=str).fillna('')
        if 'page' in q.columns:
            role_manifest = q.copy()
            role_manifest_path = p
            break

ROLE_COLUMNS = [
    'hard_negative','is_hard_negative','hard_negative_flag',
    'sparse_print','is_sparse_print','negative_control','control_page',
    'page_role','dataset_role','role','cohort_role'
]
explicit_columns = [c for c in ROLE_COLUMNS if role_manifest is not None and c in role_manifest.columns]
print('Role manifest:', role_manifest_path)
print('Explicit role columns:', explicit_columns)

def truthy(v):
    return str(v).strip().lower() in {'true','1','yes','y','hard_negative','hard-negative','negative_control','control'}

def explicit_role(row):
    if role_manifest is None or not explicit_columns:
        return None
    page = str(row.page)
    m = role_manifest[role_manifest.page.astype(str).eq(page)]
    if len(m) != 1:
        return None
    r = m.iloc[0]
    # Named boolean-like hard-negative/control columns take precedence.
    for c in ['hard_negative','is_hard_negative','hard_negative_flag','negative_control','control_page']:
        if c in explicit_columns and truthy(r[c]):
            return 'HARD_NEGATIVE_CONTROL'
    for c in ['sparse_print','is_sparse_print']:
        if c in explicit_columns and truthy(r[c]):
            return 'SPARSE_PRINT_CONTROL'
    for c in ['page_role','dataset_role','role','cohort_role']:
        if c in explicit_columns:
            v = str(r[c]).strip().lower()
            if 'hard' in v and 'negative' in v:
                return 'HARD_NEGATIVE_CONTROL'
            if 'sparse' in v:
                return 'SPARSE_PRINT_CONTROL'
            if v in {'development','modelling','modeling','annotation_positive','positive'}:
                return 'ANNOTATION_POSITIVE_DEVELOPMENT'
    return None

# Build one page-level role table from C2 because C2 restores the complete collar.
c2 = paired[paired.arm.eq('C2_PRESERVE_RING1_RING2')].copy()
role_rows=[]
for _,r in c2.iterrows():
    er = explicit_role(r)
    exposure = int(r.restored_pixels) > 0
    if er is not None:
        role = er
        role_basis = 'EXPLICIT_MANIFEST'
    elif exposure:
        role = 'COLLAR_EXPOSED_UNCLASSIFIED'
        role_basis = 'EXPOSURE_FALLBACK'
    else:
        role = 'ZERO_EXPOSURE_UNCLASSIFIED'
        role_basis = 'EXPOSURE_FALLBACK'
    invalid = str(r.page) in KNOWN_INVALID_PAGES
    if invalid:
        role = 'KNOWN_INVALID_PAGE' if PAGE201_POLICY != 'KEEP' else role
        role_basis = 'KNOWN_INVALID_OVERRIDE' if PAGE201_POLICY != 'KEEP' else role_basis
    role_rows.append({
        'page_key':r.page_key,'page':r.page,'source_group':r.source_group,
        'role':role,'role_basis':role_basis,'c2_restored_pixels':int(r.restored_pixels),
        'known_invalid':invalid,
    })
roles = pd.DataFrame(role_rows)
roles.to_csv(TABLES/'page_role_resolution.csv', index=False)
print(roles.groupby(['role','role_basis']).size())

## 4. Define the exposed analysis cohort and negative-control cohort

In [ ]:
# Exposed pages are defined experimentally by non-zero C2 collar restoration.
# Explicit control labels are retained, but a page cannot be both exposed and a zero-authority control.
exposed_keys = set(roles.loc[roles.c2_restored_pixels>0, 'page_key'])
zero_keys = set(roles.loc[roles.c2_restored_pixels==0, 'page_key'])
require(exposed_keys.isdisjoint(zero_keys), 'Exposure cohorts overlap.')

# Optional future exclusion of a known invalid page. Page201 currently has zero exposure,
# so exclusion should not alter point estimates; it is nevertheless reported explicitly.
invalid_keys = set(roles.loc[roles.known_invalid, 'page_key'])
if PAGE201_POLICY == 'EXCLUDE_INVALID':
    exposed_keys -= invalid_keys
    zero_keys -= invalid_keys

exposed = paired[paired.page_key.isin(exposed_keys)].copy()
zero = paired[paired.page_key.isin(zero_keys)].copy()

require(len(exposed)>0, 'No collar-exposed paired pages found.')
require((zero.restored_pixels==0).all(), 'Zero-exposure cohort contains non-zero restored pixels.')
require((zero.true_print_recovered.fillna(0)==0).all(), 'Zero-exposure cohort has true-print recovery.')
require((zero.annotation_retained.fillna(0)==0).all(), 'Zero-exposure cohort has annotation retention.')
require((zero.catastrophic_components.fillna(0)==0).all(), 'Zero-exposure cohort has catastrophes.')

cohort_summary = pd.DataFrame([
    {'cohort':'NOMINAL_PAIRED_ROSTER','pages':paired.page_key.nunique(),'source_groups':paired.source_group.nunique()},
    {'cohort':'COLLAR_EXPOSED','pages':exposed.page_key.nunique(),'source_groups':exposed.source_group.nunique()},
    {'cohort':'ZERO_EXPOSURE_CONTROLS','pages':zero.page_key.nunique(),'source_groups':zero.source_group.nunique()},
    {'cohort':'KNOWN_INVALID_PAGES','pages':len(invalid_keys),'source_groups':roles[roles.page_key.isin(invalid_keys)].source_group.nunique()},
])
cohort_summary.to_csv(TABLES/'cohort_summary.csv', index=False)
print(cohort_summary.to_string(index=False))

## 5. Recalculate C1/C2 on collar-exposed pages with page-clustered bootstrap

In [ ]:
def bootstrap_arm(g, resamples=BOOTSTRAP_RESAMPLES, seed=SEED):
    rng=np.random.default_rng(seed)
    pages=g.page_key.to_numpy()
    idx=g.set_index('page_key')
    out=[]
    for _ in range(resamples):
        q=idx.loc[rng.choice(pages,size=len(pages),replace=True)]
        tp=float(q.true_print_recovered.sum())
        fp=float(q.annotation_retained.sum())
        cat=float(q.catastrophic_components.sum())
        out.append((tp-5*fp-500*cat, fp/max(tp,1), fp/max(tp+fp,1)))
    return np.asarray(out,float)

summary_rows=[]
for arm,g in exposed.groupby('arm'):
    tp=int(g.true_print_recovered.sum())
    fp=int(g.annotation_retained.sum())
    tu=int(g.truth_uncertain_retained.sum())
    cat=int(g.catastrophic_components.sum())
    b=bootstrap_arm(g)
    summary_rows.append({
        'arm':arm,
        'exposed_pages':g.page_key.nunique(),
        'exposed_source_groups':g.source_group.nunique(),
        'true_print_recovered':tp,
        'annotation_retained':fp,
        'truth_uncertain_retained':tu,
        'false_true_ratio':fp/max(tp,1),
        'false_preservation_rate':fp/max(tp+fp,1),
        'utility':tp-5*fp-500*cat,
        'catastrophic_components':cat,
        'benefiting_pages_ge10_true':int((g.true_print_recovered>=10).sum()),
        'benefiting_page_rate':float((g.true_print_recovered>=10).mean()),
        'benefiting_source_groups':int(g.loc[g.true_print_recovered>=10,'source_group'].nunique()),
        'bootstrap_utility_95_low':float(np.quantile(b[:,0],.025)),
        'bootstrap_utility_95_high':float(np.quantile(b[:,0],.975)),
        'bootstrap_false_true_95_high':float(np.quantile(b[:,1],.975)),
        'bootstrap_false_preservation_95_high':float(np.quantile(b[:,2],.975)),
    })
exposed_summary=pd.DataFrame(summary_rows)
exposed_summary.to_csv(TABLES/'collar_exposed_arm_summary.csv',index=False)
print(exposed_summary.to_string(index=False))

## 6. Recalculate the marginal Ring 1 effect on exposed pages

In [ ]:
c1=exposed[exposed.arm.eq('C1_PRESERVE_RING2')].set_index('page_key')
c2=exposed[exposed.arm.eq('C2_PRESERVE_RING1_RING2')].set_index('page_key')
common=c1.index.intersection(c2.index)
require(len(common)==len(exposed_keys), 'C1/C2 exposed page rosters differ.')
fields=['true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components','restored_pixels']
inc=(c2.loc[common,fields]-c1.loc[common,fields]).reset_index()
meta=c2.loc[common,['page','source_group']].reset_index()
inc=meta.merge(inc,on='page_key',validate='one_to_one')
inc.to_csv(TABLES/'incremental_ring1_exposed_pages.csv',index=False)

ring1_summary={
    'exposed_pages':len(common),
    'additional_true_print':int(inc.true_print_recovered.sum()),
    'additional_annotation_retained':int(inc.annotation_retained.sum()),
    'additional_truth_uncertain':int(inc.truth_uncertain_retained.sum()),
    'incremental_utility':int(inc.utility.sum()),
    'additional_catastrophic_components':int(inc.catastrophic_components.sum()),
    'false_per_true':float(inc.annotation_retained.sum()/max(inc.true_print_recovered.sum(),1)),
    'pages_with_additional_ge10_true':int((inc.true_print_recovered>=10).sum()),
}
atomic_json(TABLES/'incremental_ring1_exposed_summary.json',ring1_summary)
print(json.dumps(ring1_summary,indent=2))

## 7. Hard-negative / zero-exposure control gate

In [ ]:
zero_page = zero.groupby(['page_key','page','source_group'],as_index=False).agg(
    arms=('arm','nunique'),
    restored_pixels=('restored_pixels','sum'),
    true_print_recovered=('true_print_recovered','sum'),
    annotation_retained=('annotation_retained','sum'),
    utility=('utility','sum'),
    catastrophic_components=('catastrophic_components','sum'),
)
zero_page=zero_page.merge(roles[['page_key','role','role_basis','known_invalid']],on='page_key',how='left',validate='one_to_one')
zero_page.to_csv(TABLES/'zero_exposure_control_pages.csv',index=False)

explicit_controls=zero_page[zero_page.role.isin(['HARD_NEGATIVE_CONTROL','SPARSE_PRINT_CONTROL'])].copy()
unclassified_controls=zero_page[zero_page.role.eq('ZERO_EXPOSURE_UNCLASSIFIED')].copy()

control_summary={
    'zero_exposure_pages':int(zero_page.page_key.nunique()),
    'zero_exposure_source_groups':int(zero_page.source_group.nunique()),
    'explicit_hard_negative_or_sparse_pages':int(explicit_controls.page_key.nunique()),
    'unclassified_zero_exposure_pages':int(unclassified_controls.page_key.nunique()),
    'known_invalid_zero_exposure_pages':int(zero_page.known_invalid.sum()),
    'restored_pixels_all_arms':int(zero_page.restored_pixels.sum()),
    'true_print_recovered':int(zero_page.true_print_recovered.sum()),
    'annotation_retained':int(zero_page.annotation_retained.sum()),
    'catastrophic_components':int(zero_page.catastrophic_components.sum()),
    'zero_intervention_gate_passed':bool((zero_page[['restored_pixels','true_print_recovered','annotation_retained','catastrophic_components']].to_numpy()==0).all()),
}
atomic_json(TABLES/'zero_exposure_control_summary.json',control_summary)
print(json.dumps(control_summary,indent=2))

## 8. Source-group summaries on the truly exposed population

In [ ]:
by_group=exposed.groupby(['source_group','arm'],as_index=False).agg(
    exposed_pages=('page_key','nunique'),
    restored_pixels=('restored_pixels','sum'),
    true_print_recovered=('true_print_recovered','sum'),
    annotation_retained=('annotation_retained','sum'),
    truth_uncertain_retained=('truth_uncertain_retained','sum'),
    utility=('utility','sum'),
    catastrophic_components=('catastrophic_components','sum'),
)
by_group['false_true_ratio']=by_group.annotation_retained/np.maximum(by_group.true_print_recovered,1)
by_group['false_preservation_rate']=by_group.annotation_retained/np.maximum(by_group.true_print_recovered+by_group.annotation_retained,1)
by_group.to_csv(TABLES/'collar_exposed_results_by_source_group.csv',index=False)
print(by_group.to_string(index=False))

## 9. Visual reporting

In [ ]:
sns.set_theme(style='whitegrid')

# Cohort composition.
plt.figure(figsize=(9,5))
q=cohort_summary[cohort_summary.cohort.isin(['COLLAR_EXPOSED','ZERO_EXPOSURE_CONTROLS','KNOWN_INVALID_PAGES'])]
sns.barplot(data=q,x='cohort',y='pages')
plt.title('Notebook 75 paired roster by experimental role')
plt.ylabel('Pages');plt.xlabel('')
plt.xticks(rotation=15)
plt.tight_layout();plt.savefig(VISUALS/'paired_roster_role_counts.png',dpi=180);plt.close()

# Exposed arm benefit/cost.
q=exposed_summary.set_index('arm').reindex(['C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2']).reset_index()
x=np.arange(len(q))
fig,ax=plt.subplots(figsize=(10,6))
ax.bar(x-.2,q.true_print_recovered,.4,label='true print recovered')
ax.bar(x+.2,q.annotation_retained,.4,label='annotation retained')
ax.set_xticks(x,['C1 Ring2','C2 Ring1+2']);ax.set_title('Collar-exposed pages only: paired clean-truth benefit and cost');ax.legend();plt.tight_layout();plt.savefig(VISUALS/'exposed_arm_benefit_cost.png',dpi=180);plt.close(fig)

# C1 page scatter on exposed pages.
c1e=exposed[exposed.arm.eq('C1_PRESERVE_RING2')].copy()
plt.figure(figsize=(9,7))
sns.scatterplot(data=c1e,x='true_print_recovered',y='annotation_retained',hue='source_group',size='restored_pixels',sizes=(25,260),legend=False,alpha=.8)
mx=max(c1e.true_print_recovered.max(),c1e.annotation_retained.max(),1)
plt.plot([0,mx],[0,mx],'k--',linewidth=1)
plt.title('C1 Ring2 on collar-exposed pages')
plt.tight_layout();plt.savefig(VISUALS/'c1_exposed_page_tradeoff.png',dpi=180);plt.close()

# C1 source-group stability.
g=by_group[by_group.arm.eq('C1_PRESERVE_RING2')].sort_values('utility')
fig,ax=plt.subplots(figsize=(11,max(6,.45*len(g))))
colors=['#c44e52' if v<0 else '#4c72b0' for v in g.utility]
ax.barh(g.source_group,g.utility,color=colors)
ax.axvline(0,color='black',linewidth=1)
ax.set_title('C1 Ring2 utility by source group — collar-exposed pages only')
plt.tight_layout();plt.savefig(VISUALS/'c1_exposed_source_group_utility.png',dpi=180);plt.close(fig)

print('Visuals:',VISUALS)

## 10. Verdict and reporting record

In [ ]:
# This notebook changes reporting only. It does not create a new challenger verdict.
record={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'notebook75a-stratified-reanalysis',
    'source_notebook75_run':str(SOURCE),
    'source_prediction_lock_id':lock75.get('lock_id'),
    'read_only_reanalysis':True,
    'arms_regenerated':False,
    'incumbent_or_challenger_changed':False,
    'page201_policy':PAGE201_POLICY,
    'nominal_paired_pages':int(paired.page_key.nunique()),
    'collar_exposed_pages':int(exposed.page_key.nunique()),
    'zero_exposure_pages':int(zero.page_key.nunique()),
    'known_invalid_pages_detected':sorted(roles.loc[roles.known_invalid,'page'].astype(str).unique().tolist()),
    'role_manifest':str(role_manifest_path) if role_manifest_path else None,
    'explicit_role_columns':explicit_columns,
    'zero_intervention_gate':control_summary,
    'collar_exposed_arm_results':exposed_summary.to_dict('records'),
    'ring1_incremental_result':ring1_summary,
    'interpretation':'Use COLLAR_EXPOSED for Ring1/Ring2 clustered inference. Use explicit hard-negative/sparse controls as a separate zero-intervention gate. Do not reinterpret zero-exposure rows as challenger failures.',
    'promotion_authority':False,
}
atomic_json(OUT/'notebook75a_verdict.json',record)

lines=[
    'NOTEBOOK 75A — COLLAR-EXPOSED AND HARD-NEGATIVE STRATIFIED REANALYSIS','',
    f'Source Notebook 75: {SOURCE}',
    f'Nominal paired pages: {paired.page_key.nunique()}',
    f'Collar-exposed pages: {exposed.page_key.nunique()}',
    f'Zero-exposure control pages: {zero.page_key.nunique()}',
    f'Explicit control-role columns found: {explicit_columns}',
    f'Page201 policy: {PAGE201_POLICY}','',
    'COLLAR-EXPOSED ARM SUMMARY',exposed_summary.to_string(index=False),'',
    'ZERO-EXPOSURE CONTROL GATE',json.dumps(control_summary,indent=2),'',
    'MARGINAL RING1 OVER RING2',json.dumps(ring1_summary,indent=2),'',
    'INTERPRETATION',
    'Notebook 75 outputs are unchanged. Collar-exposed pages are the correct page-clustered denominator for C1/C2 evidence. Legitimate hard-negative/sparse pages are a separate zero-intervention control. Zero-exposure pages without an explicit historical role remain labelled unclassified rather than being silently called hard negatives.'
]
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines))

## 11. Final upload package

In [ ]:
artefacts=[
    OUT/'executive_summary.txt',
    OUT/'notebook75a_verdict.json',
    AUDIT/'source_notebook75_identity.json',
    TABLES/'cohort_summary.csv',
    TABLES/'page_role_resolution.csv',
    TABLES/'collar_exposed_arm_summary.csv',
    TABLES/'incremental_ring1_exposed_pages.csv',
    TABLES/'incremental_ring1_exposed_summary.json',
    TABLES/'zero_exposure_control_pages.csv',
    TABLES/'zero_exposure_control_summary.json',
    TABLES/'collar_exposed_results_by_source_group.csv',
    VISUALS/'paired_roster_role_counts.png',
    VISUALS/'exposed_arm_benefit_cost.png',
    VISUALS/'c1_exposed_page_tradeoff.png',
    VISUALS/'c1_exposed_source_group_utility.png',
]
for p in artefacts:
    require(p.exists(),f'Missing upload artefact: {p}')
    shutil.copy2(p,UPLOAD/p.name)

manifest=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():
        manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text(
    'Notebook 75A is a read-only stratified reanalysis of Notebook 75. '
    'Start with executive_summary.txt, collar_exposed_arm_summary.csv, '
    'zero_exposure_control_summary.json, and c1_exposed_source_group_utility.png.\n'
)
zip_path=UPLOAD/'notebook75a_stratified_reanalysis_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p != zip_path:
            z.write(p,p.name)
print('Final bundle:',zip_path)

## Reading Notebook 75A

- **`COLLAR_EXPOSED`** is the correct population for C1/C2 clustered inference because those pages actually experienced a challenger intervention.
- **Explicit hard-negative/sparse controls** remain in the evidence base, but as a separate zero-intervention gate.
- **`ZERO_EXPOSURE_UNCLASSIFIED`** means historical role metadata was not found. Do not silently relabel those pages.
- If Page201 is known to be corrupt, use `PAGE201_POLICY = "EXCLUDE_INVALID"` only for this reanalysis and retain the explicit exclusion in the 75A record. Do not rewrite Notebook 75 history.
- Notebook 75A does not promote C1 or C2 and does not change `v8_8_operator.py`.